# 05 — LangGraph integration with `MemtomemStore`

`memtomem.integrations.langgraph.MemtomemStore` wraps the Python API with
a simple async interface that is easy to call from inside LangGraph nodes.
In this notebook you will:

1. Build a `MemtomemStore` pointed at a temp directory (so the notebook
   does not touch your real memtomem install).
2. Seed it with a couple of research-style memories via `store.add()`.
3. Define two LangGraph nodes: one that searches memtomem, one that writes
   findings back.
4. Compile and run a minimal two-node graph.

## Prerequisites

- **memtomem** installed:
  ```bash
  # From PyPI
  uv pip install "memtomem[ollama]" jupyter ipykernel
  # Or from a source checkout
  uv pip install -e "packages/memtomem[all]" jupyter ipykernel
  ```
- Ollama running with `nomic-embed-text`.
- `langgraph` installed:
  ```bash
  uv pip install langgraph
  ```

In [1]:
# Verify Ollama is reachable before doing anything else.
import httpx

try:
    r = httpx.get("http://localhost:11434/api/tags", timeout=2)
    r.raise_for_status()
except Exception as e:
    raise RuntimeError(
        "Ollama is not reachable at http://localhost:11434.\n"
        "Start it and pull the default embedding model:\n"
        "  ollama serve\n"
        "  ollama pull nomic-embed-text\n"
        "then re-run this cell."
    ) from e

print("Ollama is up.")

Ollama is up.


In [2]:
# Check langgraph is importable before we start building anything.
try:
    import langgraph  # noqa: F401
    from langgraph.graph import StateGraph, END
except ImportError as e:
    raise ImportError(
        "langgraph is required for this notebook. Install with:\n  uv pip install langgraph"
    ) from e

print("langgraph is importable")

langgraph is importable


## Step 1 — Build a temp-scoped `MemtomemStore`

`MemtomemStore` lazily initialises on the first call. We pass
`config_overrides` so the sqlite file lands in a temp directory instead of
`~/.memtomem/memtomem.db`. We also monkey-patch `load_config_overrides`
the same way as the other notebooks, so the user's real config file does
not leak in.

In [3]:
import json
import os
import tempfile
from pathlib import Path

import memtomem.config as _cfg
from memtomem.integrations.langgraph import MemtomemStore

tmp = tempfile.TemporaryDirectory()
tmp_path = Path(tmp.name)
db_path = tmp_path / "memory.db"
notes_dir = tmp_path / "notes"
notes_dir.mkdir()

os.environ["MEMTOMEM_STORAGE__SQLITE_PATH"] = str(db_path)
os.environ["MEMTOMEM_INDEXING__MEMORY_DIRS"] = json.dumps([str(notes_dir)])
os.environ["MEMTOMEM_EMBEDDING__PROVIDER"] = "ollama"
os.environ["MEMTOMEM_EMBEDDING__MODEL"] = "nomic-embed-text"
os.environ["MEMTOMEM_EMBEDDING__DIMENSION"] = "768"

store = MemtomemStore(
    config_overrides={
        "storage": {"sqlite_path": db_path},
        "indexing": {"memory_dirs": [notes_dir]},
        "embedding": {"provider": "ollama", "model": "nomic-embed-text", "dimension": 768},
    }
)

_orig_loader = _cfg.load_config_overrides
_cfg.load_config_overrides = lambda c: None
try:
    await store._ensure_init()
finally:
    _cfg.load_config_overrides = _orig_loader

print("store ready:", type(store).__name__)

store ready: MemtomemStore


## Step 2 — Seed the store with `store.add()`

`store.add()` appends an entry to a dated markdown file in the memory
directory and indexes it on the fly. Each call returns the file path plus
the number of chunks that were written.

In [4]:
seeds = [
    {
        "content": (
            "BM25 performs well on short factual queries but degrades on "
            "long narrative queries where dense retrievers dominate."
        ),
        "tags": ["research", "retrieval"],
    },
    {
        "content": (
            "Cross-encoder reranking typically buys 5-10 points of "
            "recall@10 at the cost of an extra second of latency."
        ),
        "tags": ["research", "reranking"],
    },
    {
        "content": (
            "Time-decay scoring is most useful for memories that become "
            "stale, such as deployment status notes or oncall handoffs."
        ),
        "tags": ["research", "decay"],
    },
]

for seed in seeds:
    result = await store.add(content=seed["content"], tags=seed["tags"])
    print(f"added -> {result['file']} ({result['indexed_chunks']} chunk(s))")

[04/11/26 11:06:26] INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=314015;file:///Users/pdstudio/.cache/uv/archive-v0/SnIIJTSJo_3oskwjndlKd/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=59710;file:///Users/pdstudio/.cache/uv/archive-v0/SnIIJTSJo_3oskwjndlKd/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

added -> /private/var/folders/4n/mt3km5rs1bn_b0w_d6hvlmhc0000gn/T/tmpyhosejy5/notes/2026-04-11.md (1 chunk(s))


                    INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=869413;file:///Users/pdstudio/.cache/uv/archive-v0/SnIIJTSJo_3oskwjndlKd/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=199876;file:///Users/pdstudio/.cache/uv/archive-v0/SnIIJTSJo_3oskwjndlKd/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

added -> /private/var/folders/4n/mt3km5rs1bn_b0w_d6hvlmhc0000gn/T/tmpyhosejy5/notes/2026-04-11.md (1 chunk(s))


                    INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=977073;file:///Users/pdstudio/.cache/uv/archive-v0/SnIIJTSJo_3oskwjndlKd/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=163082;file:///Users/pdstudio/.cache/uv/archive-v0/SnIIJTSJo_3oskwjndlKd/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

added -> /private/var/folders/4n/mt3km5rs1bn_b0w_d6hvlmhc0000gn/T/tmpyhosejy5/notes/2026-04-11.md (1 chunk(s))


## Step 3 — Define LangGraph nodes

A LangGraph node is an async function that takes the current state dict and
returns a partial state update. Our `research_node` reads the `query` from
state, runs a hybrid search through the store, and stashes the results on
`context`. Our `save_node` takes the agent's final `answer` and persists
it back into memtomem.

In [5]:
from typing import TypedDict


class AgentState(TypedDict, total=False):
    query: str
    context: list
    answer: str
    saved_to: str


async def research_node(state: AgentState) -> AgentState:
    results = await store.search(query=state["query"], top_k=3)
    # store.search returns a list of plain dicts (see MemtomemStore.search).
    return {"context": results}


def synthesize(state: AgentState) -> AgentState:
    # A trivial "agent" step: concatenate the top snippets into an answer.
    # In a real LangGraph agent this would be an LLM call.
    snippets = [c["content"] for c in state.get("context", [])]
    answer = "Findings from memory:\n" + "\n".join(f"- {s}" for s in snippets)
    return {"answer": answer}


async def save_node(state: AgentState) -> AgentState:
    finding = f"Synthesis for '{state['query']}':\n{state['answer']}"
    result = await store.add(content=finding, tags=["synthesis", "graph-output"])
    return {"saved_to": result["file"]}


print("AgentState + three node callables defined")

AgentState + three node callables defined


## Step 4 — Compile a minimal graph

Using a `TypedDict` as the state type lets LangGraph merge each node's
partial return value into the accumulated state, so later nodes can see
the fields that earlier nodes added.

In [6]:
graph = StateGraph(AgentState)
graph.add_node("research", research_node)
graph.add_node("synthesize", synthesize)
graph.add_node("save", save_node)

graph.set_entry_point("research")
graph.add_edge("research", "synthesize")
graph.add_edge("synthesize", "save")
graph.add_edge("save", END)

agent = graph.compile()
print("graph compiled:", agent)

graph compiled: <langgraph.graph.state.CompiledStateGraph object at 0x11097fa10>


## Step 5 — Run the agent

`agent.ainvoke()` runs the graph top to bottom and returns the final state.
You can see the retrieved context, the synthesized answer, and the path
of the file that the `save` node wrote back into memtomem.

In [7]:
final_state = await agent.ainvoke({"query": "when does BM25 beat dense retrieval"})

print("context chunks retrieved:", len(final_state.get("context", [])))
for c in final_state.get("context", []):
    print(f"  score={c['score']:.3f}  tags={c.get('tags')}")

print()
print("answer:")
print(final_state["answer"])

print()
print("persisted to:", final_state["saved_to"])

                    INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=96859;file:///Users/pdstudio/.cache/uv/archive-v0/SnIIJTSJo_3oskwjndlKd/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=876114;file:///Users/pdstudio/.cache/uv/archive-v0/SnIIJTSJo_3oskwjndlKd/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=936218;file:///Users/pdstudio/.cache/uv/archive-v0/SnIIJTSJo_3oskwjndlKd/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=263081;file:///Users/pdstudio/.cache/uv/archive-v0/SnIIJTSJo_3oskwjndlKd/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

context chunks retrieved: 1
  score=0.033  tags=[]

answer:
Findings from memory:
- > created: 2026-04-11T02:06:26+00:00
tags: ['research', 'retrieval']

BM25 performs well on short factual queries but degrades on long narrative queries where dense retrievers dominate.

> created: 2026-04-11T02:06:26+00:00
tags: ['research', 'reranking']

Cross-encoder reranking typically buys 5-10 points of recall@10 at the cost of an extra second of latency.

> created: 2026-04-11T02:06:26+00:00
tags: ['research', 'decay']

Time-decay scoring is most useful for memories that become stale, such as deployment status notes or oncall handoffs.

persisted to: /private/var/folders/4n/mt3km5rs1bn_b0w_d6hvlmhc0000gn/T/tmpyhosejy5/notes/2026-04-11.md


## Mapping to other LangGraph patterns

- **Checkpoints**: LangGraph's checkpointing layer is orthogonal to
  memtomem. Use memtomem for long-lived, search-ready memory and
  LangGraph's checkpoint saver for the graph execution state.
- **Tool nodes**: Wrap `store.search` and `store.add` in LangChain
  `BaseTool` subclasses if you want the agent to decide when to retrieve.
- **Sessions**: For episodic memory, call `store.start_session()` at the
  beginning of a run and `store.end_session()` at the end. Events can be
  logged with `store.log_event()` from inside any node.

## Cleanup

In [8]:
await store.close()
tmp.cleanup()

for key in (
    "MEMTOMEM_STORAGE__SQLITE_PATH",
    "MEMTOMEM_INDEXING__MEMORY_DIRS",
    "MEMTOMEM_EMBEDDING__PROVIDER",
    "MEMTOMEM_EMBEDDING__MODEL",
    "MEMTOMEM_EMBEDDING__DIMENSION",
):
    os.environ.pop(key, None)

print("clean.")

clean.
